In [32]:
import pandas as pd
import numpy as np
import ast

In [33]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [34]:
movies = movies.merge(credits, on='title')

In [35]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [6]:
movies.dropna(inplace=True)

In [7]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

In [8]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [9]:
def convert_cast(text):
    L = []
    for i in ast.literal_eval(text)[:3]:
        L.append(i['name'])
    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [10]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [11]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])

In [12]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [13]:
new_df = movies[['movie_id','title','tags']].copy()

In [14]:
new_df = movies[['movie_id','title','tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [15]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

In [17]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
similarity = cosine_similarity(vectors)

In [20]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [21]:
recommend('Avatar')

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [22]:
def fetch_poster(movie_id):
    return "https://via.placeholder.com/300x450?text=No+Image"

In [23]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]

    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    names = []
    posters = []

    for i in movies_list:
        movie_id = new_df.iloc[i[0]].movie_id
        
        names.append(new_df.iloc[i[0]].title)
        posters.append(fetch_poster(movie_id))

    return names, posters

In [24]:
names, posters = recommend('Avatar')

print(names)
print(posters)

['Titan A.E.', 'Small Soldiers', 'Independence Day', "Ender's Game", 'Aliens vs Predator: Requiem']
['https://via.placeholder.com/300x450?text=No+Image', 'https://via.placeholder.com/300x450?text=No+Image', 'https://via.placeholder.com/300x450?text=No+Image', 'https://via.placeholder.com/300x450?text=No+Image', 'https://via.placeholder.com/300x450?text=No+Image']


In [25]:
api_key ="1aec4fa452fc4a76978a0787cfa56b37"

In [26]:
import requests

In [30]:
def fetch_poster(movie_id):
    api_key = "1aec4fa452fc4a76978a0787cfa56b37"
    
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}"
    data = requests.get(url).json()
    
    print(data)   
    
    if data.get('poster_path') is None:
        return "https://via.placeholder.com/300x450?text=No+Image"
    
    return "https://image.tmdb.org/t/p/w500/" + data['poster_path']

In [31]:
names, posters = recommend('Avatar')

print(names)
print(posters)

{'adult': False, 'backdrop_path': '/t0VWB5Kc5o38w5YYgzReArrGiZ8.jpg', 'belongs_to_collection': None, 'budget': 75000000, 'genres': [{'id': 16, 'name': 'Animation'}, {'id': 878, 'name': 'Science Fiction'}, {'id': 12, 'name': 'Adventure'}, {'id': 10751, 'name': 'Family'}, {'id': 28, 'name': 'Action'}], 'homepage': '', 'id': 7450, 'imdb_id': 'tt0120913', 'origin_country': ['US'], 'original_language': 'en', 'original_title': 'Titan A.E.', 'overview': "A young man finds out that he holds the key to restoring hope and ensuring survival for the human race, while an alien species called the Drej are bent on mankind's destruction.", 'popularity': 4.3263, 'poster_path': '/el2iHk3LTJWfEnwrvcRkvWY501G.jpg', 'production_companies': [{'id': 11050, 'logo_path': None, 'name': 'David Kirschner Productions', 'origin_country': 'US'}, {'id': 211613, 'logo_path': None, 'name': 'Gary Goldman Productions', 'origin_country': ''}, {'id': 25, 'logo_path': '/qZCc1lty5FzX30aOCVRBLzaVmcp.png', 'name': '20th Centur

In [38]:
import pickle

pickle.dump(new_df, open('movies.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))